# Selector — hành vi và giá trị của learned weighting

**Cập nhật 19/10/2026:** endpoint ablation sáu C-only job đã COMPLETE và output đã kiểm tra.
**Mặc định:** CPU, đọc kết quả đã lưu; không huấn luyện, resume hoặc chạy test inference.

Phần đầu giữ hồ sơ weighting/source trước đây. Phần 6 ở cuối trình bày kết quả M4 và endpoint ablation riêng.
Chẩn đoán trajectory **G/head 7.500→8.000** vẫn chưa có report hoàn tất; đây là bước lịch sử khác, không chặn kết luận endpoint hiện tại.
Bản này cập nhật notebook local; không tự sửa notebook đang lưu trên Kaggle hoặc các Release/ZIP cũ.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / 'manifest.json').is_file() and (ROOT.parent / 'manifest.json').is_file():
    ROOT = ROOT.parent
elif not (ROOT / 'manifest.json').is_file() and (ROOT / 'KLTN_clean' / 'manifest.json').is_file():
    ROOT = ROOT / 'KLTN_clean'
if not (ROOT / 'manifest.json').is_file():
    raise FileNotFoundError('Project root not found. Open the extracted project folder.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.audit import (verify_sources, scan, zip_listing, zip_job_contracts,
                       read_test_summary, read_test_by_seed, npz_headers,
                       source_functions, function_source, statuses, export_json)

# Optional directories containing original ZIPs and data.
import json
from src.audit import resolve_inputs
INPUT_CONFIG = resolve_inputs(ROOT)
INPUTS = INPUT_CONFIG['directories']
from src.results import (read_csv_file_or_zip, metrics_by_seed,
                         test_paired_deltas, recompute_summary,
                         check_against_stored, weighting_tables)
print('Project:', ROOT)

Project: C:\Users\ADMIN\OneDrive\Tài liệu\ChatGPT\KLTN\KLTN_clean


## Source lock

In [2]:
checks = verify_sources(ROOT)
print('Frozen source files:', len(checks), '| PASS:', sum(x['status']=='PASS' for x in checks))

Frozen source files: 15 | PASS: 15


## Source

Ba luồng phân tích tồn tại: weighting, endpoint, endgame. Không gọi train hoặc forward mặc định.

In [3]:
for n in ('weighting.py','endpoint.py','endgame.py'):
    p=ROOT/'src/frozen'/n
    print(n, '| functions:',len(source_functions(p)))

weighting.py | functions: 33
endpoint.py | functions: 37
endgame.py | functions: 34


## Bằng chứng

Khai báo ZIP learned weighting đã hoàn thành khi có. Không dùng output failed-input như PASS.

In [4]:
WEIGHTING_OUTPUT=INPUT_CONFIG['weighting']  # Original 3-seed analysis ZIP
ENDGAME_OUTPUT=INPUT_CONFIG['endgame']
for label,path in [('weighting',WEIGHTING_OUTPUT),('endgame',ENDGAME_OUTPUT)]:
    if path:
        members=zip_listing(path)
        print(label, len(members),'members')
        for m in members[:20]:print(' ',m)
    else:print(label,': SKIP (no original ZIP)')

weighting 100 members
  original/REPORT.md
  original/analysis_manifest.json
  original/checkpoint_discovery.json
  original/class_mapping.csv
  original/common_probe_inputs.npz
  original/figures/class_seed_diagnostics.png
  original/figures/neighbor_weight_distributions.png
  original/figures/weight_vs_patch_noise.png
  original/global_support_usage.csv
  original/input_inventory.json
  original/matched_seed_pair_deltas.csv
  original/metric_definitions.json
  original/metric_unit_checks.json
  original/neighbor_rank_summary.csv
  original/neighbor_weight_histograms.csv
  original/output_file_hashes.json
  original/per_anchor_metrics.csv
  original/per_class_metrics.csv
  original/per_seed_equal_class_summary.csv
  original/seed_42/actual_head_forward_verification.json
endgame : SKIP (no original ZIP)


## Trạng thái chẩn đoán trajectory lịch sử

Đủ sáu checkpoint G/head đầu vào không có nghĩa đã chạy forward 7.500→8.000 thành công.
Giữ trạng thái PENDING của phép chẩn đoán đó. Endpoint ablation sáu C mới đã hoàn tất và được trình bày ở Phần 6 bên dưới.
Các checkpoint C 7.500/8.000 của endpoint không thay thế chẩn đoán trajectory G/head M4.

In [5]:
print('8000 weighting: COMPLETE_3_SEEDS; original output packaged and configured')
print('G/head 7500→8000: PENDING historical forward report; not an automatic next action')
print('Six-C endpoint ablation: COMPLETE; see Phase 6 below; no new training or inference here')

8000 weighting: COMPLETE_3_SEEDS; original output packaged and configured
G/head 7500→8000: PENDING historical forward report; not an automatic next action
Six-C endpoint ablation: COMPLETE; see Phase 6 below; no new training or inference here


## Phân tích endpoint trên CPU

Chỉ đọc bảng đã ghi; không tạo logits/weights mới và không sử dụng validation/test để chỉnh selector.

In [6]:
if WEIGHTING_OUTPUT:
    tables=weighting_tables(WEIGHTING_OUTPUT)
    for name, rows in tables.items():
        print(name, '| rows:',len(rows))
        for row in rows[:3]:print(' ',row)
else:
    print('SKIP: endpoint output not attached')

per_seed_equal_class_summary.csv | rows: 3
  {'model_seed': '42', 'entropy_mean_nats__mean': '2.6913008505633046e-07', 'effective_neighbors_mean__mean': '1.0000002691401253', 'maximum_weight_mean__mean': '0.999999982599618', 'noise_jsd_nats__mean': '1.198560656467388e-08', 'weight_noise_rms_l2__mean': '3.623918640503854e-08', 'patch_noise_rms_per_coordinate__mean': '1.3543558391554078e-09', 'uniform_displacement_rms_per_coordinate__mean': '0.030775429576190595'}
  {'model_seed': '43', 'entropy_mean_nats__mean': '9.891458377533197e-07', 'effective_neighbors_mean__mean': '1.0000009895189161', 'maximum_weight_mean__mean': '0.999999919698016', 'noise_jsd_nats__mean': '6.3596907902488e-08', 'weight_noise_rms_l2__mean': '1.8655001391596222e-07', 'patch_noise_rms_per_coordinate__mean': '6.736649441315732e-09', 'uniform_displacement_rms_per_coordinate__mean': '0.030704531941466353'}
  {'model_seed': '44', 'entropy_mean_nats__mean': '3.5376662964862715e-07', 'effective_neighbors_mean__mean': '1

## Giới hạn

Không thay đổi source huấn luyện đã chạy, không mở test mới và không kết luận PASS khi thiếu output gốc. Kết quả chỉ có giá trị trong phạm vi protocol B của Indian Pines.

In [7]:
import sys
assert 'tensorflow' not in sys.modules, 'Unexpected TensorFlow/model initialization'
print('CPU_ONLY_NO_MODEL_IMPORT_PASS')

CPU_ONLY_NO_MODEL_IMPORT_PASS


# Phần 6. Phân tích hành vi và giá trị của learned selector

**Trạng thái: hoàn tất phạm vi endpoint ablation đã khóa. Cập nhật ngày 19/10/2026.**

Phần này tổng hợp kết quả M4, chẩn đoán trọng số tại checkpoint cuối và một đối chứng endpoint riêng. Các kết quả được đọc từ output đã lưu; việc tổng hợp không huấn luyện hoặc chạy inference lại. Dataset chính là Indian Pines, giao thức B `50e988fae260`, 16 lớp, patch 11×11 và PCA 40 thành phần.

## 6.1. Câu hỏi và hai loại bằng chứng

Câu hỏi cơ chế được khảo sát là: **quy tắc weighting mà Learned-local V2 học được tại checkpoint cuối có đem lại lợi ích phân loại hơn quy tắc chọn một neighbor ngẫu nhiên cố định hay không?**

Hai thí nghiệm trả lời hai câu hỏi khác nhau:

| Thí nghiệm | Đối tượng so sánh | Vai trò |
|---|---|---|
| M4: bốn nhánh × ba seed | Original GAN, Matched ROS, Random-local với trọng số Dirichlet và Learned-local V2 | Đánh giá phương pháp trong quá trình huấn luyện M4 đầy đủ |
| Endpoint: hai nhánh × ba cặp seed | G/head Learned-local cuối được giữ cố định; huấn luyện C mới bằng learned weights hoặc Random-one-fixed | Đánh giá giá trị của quy tắc weighting cuối trong lịch C được ghép cặp |

Không cộng hoặc gộp chênh lệch giữa hai thí nghiệm. Random-local Dirichlet và Random-one-fixed là hai đối chứng khác nhau. Thí nghiệm endpoint không thay thế bảng M4 và không cô lập lợi ích của toàn bộ quá trình huấn luyện đối kháng.

## 6.2. Kết quả M4 làm nền cho câu hỏi cơ chế

M4 đã hoàn tất 12 classifier, tương ứng bốn nhánh và seed 42/43/44, mỗi job có 8.000 classifier updates. Kết quả dưới đây là trung bình ± độ lệch chuẩn mẫu (`ddof=1`) qua ba seed trên cùng 4.721 mẫu test.

| Phương pháp | OA (%) | AA (%) | Macro-F1 (%) | GM (%) | Kappa |
|---|---:|---:|---:|---:|---:|
| Original GAN | 87,552 ± 0,444 | 85,783 ± 1,641 | 81,663 ± 2,877 | 82,086 ± 4,027 | 0,8587 ± 0,0053 |
| Matched ROS | 87,143 ± 0,422 | 85,049 ± 0,681 | 81,485 ± 0,967 | 80,575 ± 3,269 | 0,8542 ± 0,0045 |
| Random-local, Dirichlet | 88,018 ± 1,199 | 86,016 ± 1,062 | 82,226 ± 3,004 | 81,862 ± 3,919 | 0,8644 ± 0,0134 |
| Learned-local V2 | 88,251 ± 1,229 | 86,046 ± 1,801 | 83,009 ± 3,015 | 81,793 ± 3,888 | 0,8668 ± 0,0139 |

V2 có Macro-F1 trung bình cao hơn Original GAN 1,345 điểm phần trăm, Matched ROS 1,524 điểm và Random-local 0,783 điểm. Tuy nhiên, V2 chỉ thắng Random-local ở hai trong ba seed; tại seed 43, V2 thấp hơn cả ba đối chứng về Macro-F1. V2 cũng không đứng đầu GM. Vì vậy, bảng M4 hỗ trợ nhận định cải thiện trung bình trong cấu hình đã kiểm tra, chưa hỗ trợ nhận định vượt trội ổn định ở mọi seed hoặc mọi độ đo.

Phân tích lớp trước đây còn cho thấy Oats, với 10 mẫu test, đóng góp khoảng 0,685 trong tổng 0,783 điểm tăng Macro-F1 so với Random-local. Kết quả này được giữ nguyên trong diễn giải; không loại lớp hoặc seed bất lợi để làm đẹp bảng. Môi trường train M4 seed 42 khác phiên bản tf-keras/NumPy của seed 43/44, nên SD không được diễn giải như chỉ phản ánh một nguồn ngẫu nhiên duy nhất.

## 6.3. Hành vi trọng số tại checkpoint 8.000

Với anchor \(x_a\), ba neighbor cùng lớp và \(\rho=0,2\), V2 sinh mẫu theo:

\[
\tilde x(a,z,c)=0,8x_a+0,2\sum_{j\in N_c(a)}w_j(z,c,a)x_j.
\]

G tạo features từ noise và điều kiện lớp. Dense head tạo logits trên các supports của lớp, sau đó graph của anchor giới hạn ba logits tham gia softmax. Anchor tác động qua graph và phép trộn; G/head không nhận trực tiếp feature của anchor làm điều kiện học.

Chẩn đoán cuối đã đo đủ ba seed, 15 learned heads mỗi seed, tổng 4.138 anchor và 64 noise mỗi anchor. Lớp đa số không có learned head. Trong toàn bộ 794.496 bộ trọng số đã khảo sát, trọng số lớn nhất đều vượt 0,999; không anchor nào đổi neighbor trội qua 64 noise. Logits vẫn thay đổi theo noise nhưng softmax rất tập trung, khiến biến thiên patch ở endpoint rất nhỏ.

Đây là mô tả của các checkpoint cuối và tập noise đã khảo sát. Không suy thành mọi noise, không xác định thời điểm hiện tượng xuất hiện và không gọi đó là bằng chứng nguyên nhân seed 43 kém. Đầu ra vẫn gồm 80% anchor và 20% tổ hợp neighbor; kết quả không có nghĩa sao chép nguyên một patch hoặc mọi mẫu sinh đều giống nhau.

Quan sát này dẫn tới đối chứng endpoint: nếu cơ chế gần chọn một neighbor, cần so giá trị của lựa chọn học được với một lựa chọn ngẫu nhiên cố định trước khi sửa noise, nhiệt độ softmax hoặc kiến trúc.

## 6.4. Thiết kế endpoint ablation

Dùng G và 15 Dense heads từ native final checkpoint M4 Learned-local tại 8.000 C updates của từng seed 42/43/44. Các mô hình sinh được khôi phục đúng source/tensor hashes, giữ cố định và gọi với `training=False`.

Với mỗi G seed, huấn luyện hai classifier mới, C seed tương ứng, trên final train 5.528 mẫu:

- **Learned weights:** dùng trực tiếp trọng số softmax thật của checkpoint trong công thức V2.
- **Random-one-fixed:** với mỗi anchor, chọn đều một trong ba neighbor bằng RNG riêng trước training, rồi giữ lựa chọn đó cố định trong cả job. Patch bổ sung là \(0,8x_a+0,2x_{r(a)}\).

Tổng cộng sáu C-only job. Hai nhánh trong mỗi cặp có cùng khởi tạo C và Adam slots, dữ liệu, graph, real IDs, fake-class schedule, anchors, noise và dropout. C dùng MSE, legacy Adam với learning rate \(2\times10^{-4}\), \(\beta_1=0,5\), 8.000 updates; global Adam clock cuối là 136.000 theo lịch M4. Những lượt D/G bị bỏ chỉ tăng iteration counter, không dùng zero-gradient updates. G/head không được huấn luyện lại.

Đây là đối chứng toàn bộ quy tắc weighting endpoint: Learned dùng soft weights thật, Random-one-fixed dùng một neighbor. Thiết kế chưa tách riêng hiệu ứng danh tính neighbor khỏi mức độ mềm của trọng số. Seed G/C và random map thay cùng block; ba cặp không tách được phương sai do từng nguồn.

## 6.5. Kết quả endpoint ablation

| Cặp seed G/C | Learned Macro-F1 (%) | Random-one-fixed Macro-F1 (%) | Learned − Random (điểm phần trăm) |
|---|---:|---:|---:|
| 42 | 82,512 | 82,857 | −0,345 |
| 43 | 83,510 | 78,067 | +5,442 |
| 44 | 80,466 | 78,429 | +2,038 |
| Trung bình ± SD mẫu | **82,163 ± 1,551** | **79,784 ± 2,667** | **+2,379** |

SD mẫu của ba paired differences là 2,909 điểm phần trăm. Learned thắng hai trong ba cặp; seed 42 thua. Ba cặp là đơn vị lặp của thí nghiệm; không coi 4.721 pixel test như 4.721 lần huấn luyện độc lập.

| Độ đo | Learned: TB ± SD mẫu | Random-one-fixed: TB ± SD mẫu |
|---|---:|---:|
| OA (%) | 87,192 ± 0,903 | 86,705 ± 1,661 |
| AA (%) | 85,263 ± 1,221 | 83,439 ± 1,854 |
| Macro-F1 (%) | 82,163 ± 1,551 | 79,784 ± 2,667 |
| GM (%) | 79,907 ± 4,580 | 76,269 ± 1,454 |
| Kappa | 0,854850 ± 0,010282 | 0,849095 ± 0,018714 |
| NLL ↓ | 1,034228 ± 0,185781 | 0,993130 ± 0,128543 |
| Brier ↓ | 0,225038 ± 0,017608 | 0,228192 ± 0,028805 |

GM được bổ sung từ recall 16 lớp trong dự đoán đã lưu. NLL dùng đúng clipping \(10^{-12}\) của runner. Learned có NLL trung bình kém hơn; kết quả không hỗ trợ nhận định cải thiện mọi mặt hoặc hiệu chuẩn xác suất tốt hơn.

Ở seed 43, F1 Learned cao hơn ở 14/16 lớp, hòa một và thấp hơn một. Lớp Buildings–Grass–Trees–Drives có F1 56,180% so với 15,267%, recall 50/99 so với 10/99. Riêng lớp này đóng góp 2,558 trong tổng 5,442 điểm Macro-F1 tăng, khoảng 47%; phần còn lại vẫn dương.

Ở seed 42, Random-one-fixed đúng riêng 158 mẫu và Learned đúng riêng 54 mẫu; OA Learned giảm 2,203 điểm phần trăm, trong khi Macro-F1 giảm 0,345 điểm. Ở seed 44, Learned đúng riêng 212 mẫu so với 176 của Random; Macro-F1 tăng 2,038 điểm. Các số đếm đúng riêng mô tả vị trí khác biệt, không xác định nguyên nhân mô hình học được hoặc mắc lỗi.

Endpoint seed 43 có lợi không giải thích hoặc đảo ngược kết quả seed 43 trong M4 cũ: classifier và quá trình huấn luyện khác nhau. Không gán chênh lệch M4 cũ cho một selector hỏng từ các quan sát này.

## 6.6. Kết luận và giới hạn

> Trong M4 trên Indian Pines theo Protocol B, Learned-local V2 đạt Macro-F1 trung bình 83,009 ± 3,015%, cao hơn Random-local Dirichlet 0,783 điểm phần trăm và thắng ở hai trong ba seed. Chẩn đoán cuối cho thấy trọng số tập trung gần một neighbor và ít thay đổi theo noise. Trong endpoint ablation riêng, giữ cố định G/head và huấn luyện sáu classifier mới theo lịch ghép cặp, learned weighting đạt 82,163 ± 1,551% Macro-F1, so với 79,784 ± 2,667% của Random-one-fixed; chênh lệch trung bình +2,379 điểm phần trăm, với hai trong ba cặp có lợi. Kết quả cung cấp bằng chứng thực nghiệm cho giá trị của quy tắc weighting cuối so với đối chứng ngẫu nhiên cố định đã xét, dù chưa chứng minh lợi thế ổn định hoặc lợi ích riêng của quá trình huấn luyện đối kháng.

Test Indian Pines đã được xem trước khi thiết kế endpoint ablation, nên phần này là nghiên cứu hậu nghiệm. Protocol B có chồng lấn không gian; kết quả không chứng minh đã xử lý leakage hoặc tổng quát hóa sang vùng độc lập. Chỉ ba cặp và biến thiên giữa seed còn đáng kể; không tuyên bố ý nghĩa thống kê hoặc tính mới thuật toán chỉ từ bảng này.

Chẩn đoán 7.500→8.000 của **G/head M4** vẫn chưa có report forward hoàn tất trong hồ sơ đã xác minh. Đó là trạng thái khác với endpoint ablation sáu C mới đã hoàn tất. Các checkpoint C tại 7.500 có trong backup hiện tại không thay thế checkpoint hoặc chẩn đoán trajectory G. Phần chưa chạy này không được dùng để suy luận bão hòa xuất hiện khi nào và không được biến thành điều kiện buộc train lại.

Phạm vi endpoint ablation được đóng sau kết quả này. Các giả thuyết về k, rho, metric neighbor, temperature hoặc noise vẫn là hướng tương lai, chưa phải cải tiến được chứng minh và không tự mở thành lượt train mới.

## 6.7. Nguồn và cách đọc hồ sơ

- M4: `outputs/bundles/03_M4_TEST_RESULTS.zip`, đủ 12 classifier; notebook dùng bảng gốc để tính mean/SD và paired differences.
- Chẩn đoán weights: `outputs/bundles/05_V2_WEIGHTING_8000.zip`, ba seed, 15 heads/seed, train-only forward đã hoàn tất trước đây.
- Endpoint analysis: `outputs/bundles/07_SELECTOR_ENDPOINT_ABLATION_ANALYSIS_20261019.zip`, gồm sáu predictions gốc, metrics, per-class, confusion matrices, báo cáo và audit.
- Backup endpoint đầy đủ: `D:/HUIT/Chương trình học/HK1 ( 2026- 2027)/KLTN - GAN/outputs/6. Selector Analysis/selector_endpoint_ablation_clean_backup.zip`; 130.717.579 bytes, SHA-256 `7dcdcbdd1a62948a704162334c992b35674cdbcffd38843e5360426b319fc062`.
- Audit endpoint đã kiểm 311 điều kiện CPU về file hashes, counters, provenance, pairing và phép tính từ predictions. Lượt tổng hợp không chạy TensorFlow, không tái kiểm numerical restore và không thay checkpoint. Gói phân tích nhẹ không chứa lại weights; giữ backup gốc để bảo toàn sáu classifier và native checkpoints.

`notebooks/selector.ipynb` hiển thị bảng và kết luận bằng cách đọc các gói đã lưu; Run All vẫn là CPU và không chạy lại training/test inference. Các ZIP đầy đủ hoặc GitHub Release đã xuất trước ngày này không tự cập nhật theo bản local mới.

## Bảng tái lập từ output đã lưu

Những cell tiếp theo chỉ đọc ZIP phân tích đã kiểm, kiểm SHA và tính lại tổng hợp từ CSV.
File dự đoán gốc và confusion matrices có trong gói; không khởi tạo C/G/D hoặc optimizer.
Audit 311 điều kiện là hồ sơ kiểm file và số liệu đã lưu, không phải 311 thí nghiệm và không phải restore TensorFlow mới.

In [8]:
import csv, hashlib, io, json, zipfile
from statistics import mean, stdev
import pandas as pd
from IPython.display import display

ENDPOINT_BUNDLE = Path(INPUT_CONFIG['endpoint_ablation_analysis'])
ENDPOINT_BUNDLE_SHA = '7fc8bb5145100d7e62326492c7e0d31dd58769fdb4e8a56b6063dd321668f96b'
assert hashlib.sha256(ENDPOINT_BUNDLE.read_bytes()).hexdigest() == ENDPOINT_BUNDLE_SHA, 'Endpoint analysis bundle SHA mismatch'
with zipfile.ZipFile(ENDPOINT_BUNDLE) as endpoint_zip:
    file_pins = json.loads(endpoint_zip.read('FILES_SHA256.json'))
    for name, pin in file_pins.items():
        assert hashlib.sha256(endpoint_zip.read(name)).hexdigest() == pin, name
    endpoint_audit = json.loads(endpoint_zip.read('AUDIT_RESULT.json'))
    assert endpoint_audit['status'] == 'PASS_SAVED_ARTIFACT_AUDIT'
    assert endpoint_audit['checks_passed'] == 311
    def endpoint_csv(name):
        return list(csv.DictReader(io.StringIO(endpoint_zip.read(name).decode('utf-8-sig'))))
    endpoint_rows = endpoint_csv('metrics_per_seed_with_GM.csv')
    endpoint_summary_saved = endpoint_csv('summary_mean_sample_SD.csv')
    endpoint_pairs_saved = endpoint_csv('paired_differences_recomputed.csv')
    endpoint_class_pairs = endpoint_csv('per_class_paired_contributions.csv')
assert {(int(r['G_seed']), r['method']) for r in endpoint_rows} == {(s,m) for s in (42,43,44) for m in ('learned_weights','random_one_fixed')}
print('COMPLETE: six saved C classifiers; exact analysis bundle and member hashes PASS')
print('Original full checkpoint backup SHA:', endpoint_audit['source_sha256'])
print('No model import, training, restore or inference performed by these cells.')


COMPLETE: six saved C classifiers; exact analysis bundle and member hashes PASS
Original full checkpoint backup SHA: 7dcdcbdd1a62948a704162334c992b35674cdbcffd38843e5360426b319fc062
No model import, training, restore or inference performed by these cells.


In [9]:
m4_rows_phase6 = metrics_by_seed(INPUT_CONFIG['test'])
m4_summary_phase6 = recompute_summary(m4_rows_phase6)
m4_pairs_phase6 = test_paired_deltas(m4_rows_phase6)
m4_display = []
for r in m4_summary_phase6:
    m4_display.append({'M4 arm': r['arm'], **{
        k + ' (%)': f"{100*r[k+'_mean']:.3f} ± {100*r[k+'_sd']:.3f}"
        for k in ('OA','AA','Macro_F1','GM')},
        'Kappa': f"{r['Kappa_mean']:.4f} ± {r['Kappa_sd']:.4f}"})
print('M4 full training comparison: Random-local uses Dirichlet mixing')
display(pd.DataFrame(m4_display))
display(pd.DataFrame(m4_pairs_phase6))


M4 full training comparison: Random-local uses Dirichlet mixing


,M4 arm,OA (%),AA (%),Macro_F1 (%),GM (%),Kappa
0,original_gan,87.552 ± 0.444,85.783 ± 1.641,81.663 ± 2.877,82.086 ± 4.027,0.8587 ± 0.0053
1,matched_ros,87.143 ± 0.422,85.049 ± 0.681,81.485 ± 0.967,80.575 ± 3.269,0.8542 ± 0.0045
2,random_local,88.018 ± 1.199,86.016 ± 1.062,82.226 ± 3.004,81.862 ± 3.919,0.8644 ± 0.0134
3,learned_local,88.251 ± 1.229,86.046 ± 1.801,83.009 ± 3.015,81.793 ± 3.888,0.8668 ± 0.0139


,seed,against,macro_f1_pp,oa_pp
0,42,original_gan,2.581231,1.588646
1,42,matched_ros,6.061667,2.986655
2,42,random_local,2.103498,0.466003
3,43,original_gan,-1.242726,-0.084728
4,43,matched_ros,-0.891705,-0.063546
5,43,random_local,-2.076199,-0.783732
6,44,original_gan,2.697651,0.593095
7,44,matched_ros,-0.599318,0.402457
8,44,random_local,2.321115,1.016734


In [10]:
endpoint_per_seed = []
for seed in (42,43,44):
    l, r = [next(v for v in endpoint_rows if int(v['G_seed']) == seed and v['method'] == method)
            for method in ('learned_weights','random_one_fixed')]
    endpoint_per_seed.append({'G/C seed': seed,
        'Learned Macro-F1 (%)': 100*float(l['Macro_F1']),
        'Random-one-fixed Macro-F1 (%)': 100*float(r['Macro_F1']),
        'Learned − Random (pp)': 100*(float(l['Macro_F1'])-float(r['Macro_F1']))})
endpoint_summary_display = []
for metric in ('OA','AA','Macro_F1','GM','Kappa','NLL','Brier'):
    l, r = [[float(v[metric]) for v in endpoint_rows if v['method'] == method]
            for method in ('learned_weights','random_one_fixed')]
    delta = [a-b for a,b in zip(l,r)]
    saved = next(v for v in endpoint_summary_saved if v['metric'] == metric)
    computed = {'learned_mean':mean(l), 'learned_sample_SD':stdev(l),
                'random_mean':mean(r), 'random_sample_SD':stdev(r),
                'paired_mean_difference':mean(delta), 'paired_sample_SD':stdev(delta)}
    assert all(abs(value-float(saved[key])) < 1e-12 for key,value in computed.items()), metric
    scale = 100 if metric in ('OA','AA','Macro_F1','GM') else 1
    endpoint_summary_display.append({'metric': metric + (' (%)' if scale == 100 else ''),
        'Learned mean ± SD': f'{scale*mean(l):.6f} ± {scale*stdev(l):.6f}',
        'Random mean ± SD': f'{scale*mean(r):.6f} ± {scale*stdev(r):.6f}',
        'Mean paired difference': scale*mean(delta),
        'Paired SD': scale*stdev(delta)})
print('Frozen endpoint + fresh C: compare with Random-one-fixed; sample SD, ddof=1')
display(pd.DataFrame(endpoint_per_seed))
display(pd.DataFrame(endpoint_summary_display))
print('Macro-F1 wins:', sum(v['Learned − Random (pp)']>0 for v in endpoint_per_seed), '/ 3 paired seeds')


Frozen endpoint + fresh C: compare with Random-one-fixed; sample SD, ddof=1
Macro-F1 wins: 2 / 3 paired seeds


,G/C seed,Learned Macro-F1 (%),Random-one-fixed Macro-F1 (%),Learned − Random (pp)
0,42,82.512133,82.856862,-0.344728
1,43,83.509510,78.067051,5.442459
2,44,80.466483,78.428506,2.037977


,metric,Learned mean ± SD,Random mean ± SD,Mean paired difference,Paired SD
0,OA (%),87.191979 ± 0.902742,86.704794 ± 1.661040,0.487185,2.563541
1,AA (%),85.262608 ± 1.221227,83.438668 ± 1.853524,1.823940,3.020287
2,Macro_F1 (%),82.162709 ± 1.551314,79.784140 ± 2.667185,2.378569,2.908588
3,GM (%),79.906651 ± 4.579978,76.269062 ± 1.453485,3.637590,5.904595
4,Kappa,0.854850 ± 0.010282,0.849095 ± 0.018714,0.005754,0.028976
5,NLL,1.034228 ± 0.185781,0.993130 ± 0.128543,0.041098,0.306824
6,Brier,0.225038 ± 0.017608,0.228192 ± 0.028805,-0.003154,0.046058


In [11]:
class43 = [v for v in endpoint_class_pairs if int(v['seed']) == 43]
differences43 = [float(v['F1_difference_pp']) for v in class43]
print('Seed 43 per-class F1: higher', sum(d>1e-12 for d in differences43),
      '| tied', sum(abs(d)<=1e-12 for d in differences43),
      '| lower', sum(d < -1e-12 for d in differences43))
display(pd.DataFrame([{'class_id':int(v['class_id']), 'class':v['class_name'],
    'test samples':int(v['support']), 'Learned F1 (%)':100*float(v['learned_F1']),
    'Random F1 (%)':100*float(v['random_F1']),
    'Delta F1 (pp)':float(v['F1_difference_pp']),
    'Contribution to Macro-F1 delta (pp)':float(v['contribution_to_macro_difference_pp'])}
    for v in class43]))
print('Per-class tables for all seeds, predictions and six confusion matrices are retained in:', ENDPOINT_BUNDLE.name)


Seed 43 per-class F1: higher 14 | tied 1 | lower 1
Per-class tables for all seeds, predictions and six confusion matrices are retained in: 07_SELECTOR_ENDPOINT_ABLATION_ANALYSIS_20261019.zip


,class_id,class,test samples,Learned F1 (%),Random F1 (%),Delta F1 (pp),Contribution to Macro-F1 delta (pp)
0,1,Alfalfa,25,86.206897,82.142857,4.064039,0.254002
1,2,Corn-notill,675,83.628632,79.154930,4.473703,0.279606
2,3,Corn-mintill,404,84.062059,83.844011,0.218048,0.013628
3,4,Corn,99,61.202186,49.799197,11.402989,0.712687
4,5,Grass-pasture,274,90.181818,89.502762,0.679056,0.042441
5,6,Grass-trees,354,98.166432,95.485636,2.680795,0.167550
6,7,Grass-pasture-mowed,12,68.571429,70.588235,-2.016807,-0.126050
7,8,Hay-windrowed,250,100.000000,100.000000,0.000000,0.000000
8,9,Oats,10,94.736842,82.352941,12.383901,0.773994
9,10,Soybean-notill,503,92.857143,89.049587,3.807556,0.237972


## Chốt phạm vi

Endpoint ablation đã COMPLETE. Learned có lợi thế Macro-F1 trung bình +2,379 điểm phần trăm,
thắng 2/3 cặp so với Random-one-fixed. Seed 42 thua; NLL trung bình kém hơn.
Đây là bằng chứng hậu nghiệm trong Protocol B; không chứng minh lợi thế ổn định, toàn bộ GAN có ích,
soft weighting hơn hard learned selection, hoặc đã xử lý spatial leakage.

Phần trajectory G/head 7.500→8.000 vẫn được ghi đúng là chưa có report hoàn tất.
Notebook không tự chuyển trạng thái này thành công việc GPU tiếp theo.

In [12]:
assert not any(name in sys.modules for name in ('tensorflow','tf_keras','keras')), 'Unexpected model runtime import'
print('PHASE6_SAVED_RESULTS_COMPLETE | CPU_ONLY | NO_TRAINING | NO_MODEL_INFERENCE')


PHASE6_SAVED_RESULTS_COMPLETE | CPU_ONLY | NO_TRAINING | NO_MODEL_INFERENCE
